In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q13/q13_rewrite/checkpoints/pre_cell_2.pickle

In [4]:
%%RecordEvent
%%time
### cell 2 ###

# 1) Count orders per customer (exclude special requests)
cust_order_counts = (
    orders
    .loc[
        ~orders["O_COMMENT"].str.contains(r"special[\S|\s]*requests"),
        "O_CUSTKEY"
    ]
    .value_counts()
)

# 2) Map counts back to every customer, fill missing with 0 and cast to int
order_counts = (
    customer["C_CUSTKEY"]
    .map(cust_order_counts)
    .fillna(0)
    .astype(int)
)

# 3) Get distribution of customers by their order count
dist = order_counts.value_counts()

# 4) Build the final DataFrame and sort by CUSTDIST desc, then C_COUNT desc
total = (
    dist
    .rename("CUSTDIST")        # name the counts column
    .reset_index()
)
# rename the columns in one go to avoid a KeyError
total.columns = ["C_COUNT", "CUSTDIST"]

# sort by customer distribution and then by count
total = total.sort_values(["CUSTDIST", "C_COUNT"], ascending=[False, False])


CPU times: user 79.5 ms, sys: 52.8 ms, total: 132 ms
Wall time: 143 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q13/rewritten/o4_mini_high_small/checkpoints/post_cell_2_try_2.pickle